In [ ]:
# Train / Caption / Eval cho Model V2 (EfficientNet/ViT + Decoder LSTM/Transformer)

Notebook này hướng dẫn đầy đủ các bước để: mount Google Drive (Colab), chuẩn bị file input (nếu cần), huấn luyện `train_v2.py` với encoder V2 (EfficientNet/ViT) và decoder LSTM hoặc Transformer, đánh giá và demo sinh caption cho 1 ảnh (hiển thị attention nếu dùng LSTM). Giữ nguyên code gốc trong project, chỉ bổ sung ô markdown và lệnh chạy tiện lợi.

## 1) Kết nối Google Drive (Chỉ Colab)
Chạy ô này để mount Drive - bắt buộc nếu bạn để dữ liệu trên Drive. Nếu chạy local, bỏ qua ô này và chỉnh `PROJECT_DIR`/`DATA_FOLDER` tương ứng.
```python
from google.colab import drive
drive.mount('/content/drive')
```

## 2) Cài dependencies cần thiết
Cài `timm`, `rouge-score`, `nltk`. `pycocoevalcap` là tùy chọn (cài phức tạp).
```python
# Cài các package cơ bản (chạy 1 lần)
!pip install --upgrade pip
!pip install timm rouge-score nltk
# Nếu muốn đánh giá CIDEr, cài pycocoevalcap (tùy chọn)
# !pip install git+https://github.com/salaniz/pycocoevalcap.git

import nltk
nltk.download('wordnet')
nltk.download('omw-1.4')
```

## 3) Thiết lập đường dẫn dự án và biến môi trường
Chỉnh `PROJECT_DIR`, `DATA_FOLDER`, `CHECKPOINT_DIR` theo Drive hoặc local của bạn.
```python
# Thay đổi nếu cần theo hệ thống của bạn
PROJECT_DIR = '/content/drive/MyDrive/Image_captioning_flickr8k/a-PyTorch-Tutorial-to-Image-Captioning'
DATA_FOLDER = '/content/drive/MyDrive/Image_captioning_flickr8k/dataset/flickr8k_processed'
DATA_NAME = 'flickr8k_5_cap_per_img_5_min_word_freq'
CHECKPOINT_DIR = '/content/drive/MyDrive/Image_captioning_flickr8k/checkpoints'
SAMPLES_DIR = '/content/drive/MyDrive/Image_captioning_flickr8k/eval_samples'

import os
os.chdir(PROJECT_DIR)
os.makedirs(CHECKPOINT_DIR, exist_ok=True)
os.makedirs(SAMPLES_DIR, exist_ok=True)
print('Project dir:', PROJECT_DIR)
print('Data folder:', DATA_FOLDER)
```

## 4) (Tuỳ chọn) Tạo các file input đã xử lý (`create_input_files.py`)
Chạy bước này nếu bạn chưa có file HDF5 / JSON trong `DATA_FOLDER` (do `create_input_files.py` tạo ra). Nếu đã có, bỏ qua.
```python
# Nếu bạn chưa có files processed, chạy create_input_files với tham số phù hợp.
# Ví dụ (chỉnh tham số theo dataset của bạn):
# !python ../create_input_files.py --input_folder /content/drive/MyDrive/... --output_folder '{DATA_FOLDER}' --num_images 6000 --max_len 50
print('Bỏ qua nếu đã có files trong', DATA_FOLDER)
```

## 5) Huấn luyện Model V2 (Ví dụ lệnh)
Bạn có thể chọn `--decoder_type lstm` (dùng decoder LSTM+Attention như gốc) hoặc `--decoder_type transformer` (transformer decoder mới).
Các tham số quan trọng: `--encoder_backbone` (vit_small_patch16_224 hoặc efficientnet_b0), `--batch_size`, `--accumulation_steps`, `--use_amp`, `--workers`, `--epochs`, `--checkpoint_dir`, `--data_folder`, `--data_name`.
```python
# Ví dụ chạy training trên Colab (chỉnh tham số theo GPU):
# Train Transformer decoder (mặc định dùng vit_small_patch16_224 nếu không đổi)
!python ../train_v2.py --encoder_backbone vit_small_patch16_224 --decoder_type transformer --batch_size 4 --accumulation_steps 2 --use_amp --workers 1 --prefetch_factor 1 --lr 4e-4 --epochs 2 --checkpoint_dir '{CHECKPOINT_DIR}' --data_folder '{DATA_FOLDER}' --data_name '{DATA_NAME}'

# Hoặc train LSTM decoder (dùng attention và có thể visualize attention sau)
# !python ../train_v2.py --encoder_backbone vit_small_patch16_224 --decoder_type lstm --batch_size 8 --lr 4e-4 --epochs 5 --checkpoint_dir '{CHECKPOINT_DIR}' --data_folder '{DATA_FOLDER}' --data_name '{DATA_NAME}'
```
Ghi chú:
- Để resume từ checkpoint, thêm `--resume /path/to/checkpoint.pth`.
- Nếu muốn chạy nhẹ trên Colab: giảm `--batch_size`, giảm `workers`, bật `--use_amp` và dùng `--accumulation_steps` để tăng hiệu quả.

### KIỂM TRA CHECKPOINTS (V2)
Liệt các file checkpoint, metadata để bạn dễ chọn file resume hoặc file BEST.
```python
import os, torch, glob, json
print('CHECKPOINT_DIR =', CHECKPOINT_DIR)
ck_files = sorted(glob.glob(os.path.join(CHECKPOINT_DIR, 'checkpoint_v2_epoch_*.pth')) )
best_file = os.path.join(CHECKPOINT_DIR, 'BEST_checkpoint_v2.pth')
if os.path.exists(best_file):
    print('Found BEST checkpoint:', best_file)
else:
    print('BEST checkpoint not found in', CHECKPOINT_DIR)
if len(ck_files) == 0:
    print('No epoch checkpoints found (pattern checkpoint_v2_epoch_*.pth)')
else:
    print('Found epoch checkpoints:')
    for p in ck_files:
        print(' -', p)

def read_ck_metadata(path):
    try:
        ck = torch.load(path, map_location='cpu', weights_only=False)
        meta = {}
        meta['path'] = path
        meta['epoch'] = ck.get('epoch', None)
        meta['best_bleu4'] = ck.get('best_bleu4', ck.get('bleu-4', None))
        meta['has_encoder'] = 'encoder' in ck
        meta['has_decoder'] = 'decoder' in ck
        return meta
    except Exception as e:
        return {'path': path, 'error': str(e)}

print('\nCheckpoint metadata:')
for p in [best_file] + ck_files:
    if os.path.exists(p):
        print('\n->', p)
        meta = read_ck_metadata(p)
        print(json.dumps(meta, indent=2))
    else:
        print('\n->', p, ' (missing)')
```

## 6) Demo sinh caption cho 1 ảnh (trực tiếp trong cell)
- Ô này hỗ trợ cả LSTM (có attention => visualize) và Transformer (không có attention heatmap).
- Yêu cầu: `BEST_checkpoint_v2.pth` hoặc `checkpoint_v2_epoch_X.pth` phải có `encoder` + `decoder` state_dict hoặc module object.
```python
import sys, os, json, torch, matplotlib.pyplot as plt
from PIL import Image

# 1) Project path
PROJECT_DIR = '/content/drive/MyDrive/Image_captioning_flickr8k/a-PyTorch-Tutorial-to-Image-Captioning'
if PROJECT_DIR not in sys.path:
    sys.path.append(PROJECT_DIR)

# 2) Paths (chỉnh khi cần)
CHECKPOINT_PATH = os.path.join(CHECKPOINT_DIR, 'BEST_checkpoint_v2.pth')
IMG_PATH = os.path.join('/content/drive/MyDrive/Image_captioning_flickr8k/dataset/Test', 'COCO_val2014_000000000241.jpg')
WORDMAP_PATH = os.path.join(DATA_FOLDER, 'WORDMAP_' + DATA_NAME + '.json')
beam_size = 3  # 1 = greedy, >1 = beam
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print('Device:', device)

# 3) Load word_map
with open(WORDMAP_PATH, 'r', encoding='utf-8') as f:
    word_map = json.load(f)
rev_word_map = {int(v): k for k, v in word_map.items()}
vocab_size = len(word_map)

# 4) Import model builder (model_v2) và helpers (eval_v2 or caption.py)
from model_v2_encoder_cnn import build_model_v2, ModelV2Config
caption_helpers_available = False
try:
    from caption import caption_image_beam_search, visualize_att  # type: ignore
    caption_helpers_available = True
except Exception:
    caption_helpers_available = False

# 5) Build empty model (config must match training)
# Chỉnh cấu hình tương ứng với checkpoint bạn dùng (encoder_dim, backbone, decoder_type)
cfg = ModelV2Config(encoder_backbone='vit_small_patch16_224', encoder_dim=512, decoder_type='lstm')
encoder, decoder = build_model_v2(vocab_size=vocab_size, config=cfg)

# 6) Load checkpoint (hỗ trợ state_dict hoặc module object)
ck = torch.load(CHECKPOINT_PATH, map_location=device, weights_only=False)
# Load encoder weights
try:
    if 'encoder' in ck and isinstance(ck['encoder'], dict):
        encoder.load_state_dict(ck['encoder'])
    elif 'encoder' in ck and hasattr(ck['encoder'], 'state_dict'):
        encoder.load_state_dict(ck['encoder'].state_dict())
    elif 'encoder' in ck and isinstance(ck['encoder'], torch.nn.Module):
        encoder = ck['encoder']
except Exception as e:
    print('Warning: failed to load encoder weights:', e)
# Load decoder weights
try:
    if 'decoder' in ck and isinstance(ck['decoder'], dict):
        decoder.load_state_dict(ck['decoder'])
    elif 'decoder' in ck and hasattr(ck['decoder'], 'state_dict'):
        decoder.load_state_dict(ck['decoder'].state_dict())
    elif 'decoder' in ck and isinstance(ck['decoder'], torch.nn.Module):
        decoder = ck['decoder']
except Exception as e:
    print('Warning: failed to load decoder weights:', e)

encoder = encoder.to(device).eval()
decoder = decoder.to(device).eval()

# 7) Image preprocessing (chỉnh resize cho ViT là 224 nếu muốn)
import torchvision.transforms as transforms
from torchvision.transforms import ToTensor, Resize, Compose
# Nếu encoder là ViT (224) hãy dùng Resize((224,224)), nếu EfficientNet bạn có thể dùng 256
resize_size = (224,224) if 'vit' in cfg.encoder_backbone else (256,256)
transform = Compose([Resize(resize_size), ToTensor(),
                     transforms.Normalize(mean=[0.485,0.456,0.406], std=[0.229,0.224,0.225])])
pil_img = Image.open(IMG_PATH).convert('RGB')
input_tensor = transform(pil_img).unsqueeze(0).to(device)

# 8) Decode: ưu tiên dùng caption_image_beam_search nếu tương thích (gốc), nếu không dùng hàm trong eval_v2.py
from eval_v2 import greedy_decode_one, beam_search_decode_one
seq_ids = None
alphas = None

if caption_helpers_available:
    try:
        seq_ids, alphas = caption_image_beam_search(encoder=encoder, decoder=decoder, image_path=IMG_PATH, word_map=word_map, beam_size=beam_size)
    except Exception as e:
        print('caption_image_beam_search failed (fallback):', e)
        caption_helpers_available = False

if not caption_helpers_available:
    if beam_size and beam_size > 1:
        seq_ids = beam_search_decode_one(encoder, decoder, input_tensor, word_map, rev_word_map, device, beam_size=beam_size, max_len=50)
    else:
        seq_ids = greedy_decode_one(encoder, decoder, input_tensor, word_map, rev_word_map, device, max_len=50)

# 9) Convert ids -> words
words = [rev_word_map.get(int(i), '<unk>') for i in seq_ids]
if words and words[0] == '<start>':
    words = words[1:]
if '<end>' in words:
    words = words[:words.index('<end>')]
caption_text = ' '.join(words)
print('Predicted caption:', caption_text)

# 10) Show image and title
%matplotlib inline
plt.figure(figsize=(6,6))
plt.imshow(pil_img)
plt.axis('off')
plt.title(caption_text)
plt.show()

# 11) Nếu có alphas và visualize_att, hiển thị attention (thường chỉ có với LSTM/Attention)
if alphas is not None:
    try:
        alphas_tensor = torch.FloatTensor(alphas)
        visualize_att(IMG_PATH, seq_ids, alphas_tensor, rev_word_map, smooth=True)
    except Exception as e:
        print('Could not visualize attention:', e)
```

## 7) Chạy đánh giá V2 (eval_v2.py)
Chạy script `eval_v2.py` để tính BLEU/ROUGE/METEOR/(CIDEr nếu có), lưu ra `SAMPLES_DIR`.
```python
# Ví dụ: chạy eval toàn tập và lưu samples
!python ../eval_v2.py --checkpoint '{CHECKPOINT_DIR}/BEST_checkpoint_v2.pth' --data_folder '{DATA_FOLDER}' --data_name '{DATA_NAME}' --beam_size 3 --save_samples_dir '{SAMPLES_DIR}' --num_samples 10

# Hiển thị ngắn gọn summary nếu có
import json, os
summary_path = os.path.join(SAMPLES_DIR, 'summary.json')
if os.path.exists(summary_path):
    with open(summary_path, 'r', encoding='utf-8') as f:
        s = json.load(f)
    print('Summary:')
    print(json.dumps(s, indent=2, ensure_ascii=False))
else:
    print('No summary.json found in', SAMPLES_DIR)
```
